# File-format reference

A cheatsheet for the file formats and pipeline conventions that come up
across the other notebooks. It's a reference, not a tutorial. Skim or
jump to whatever you need; nothing here depends on running anything
else first.

The four task notebooks have callouts pointing here when a format comes
up. If you've worked with pathogen-genomics data before, you'll know
most of this already and can skip it.

> **Kernel:** Pick **`Python 3 (Local)`** if JupyterLab prompts. Don't pick `Python 3 (ipykernel) (Local)` — it's missing libraries the other notebooks need. (This notebook is markdown-only so technically any kernel works, but be consistent with the others.)

## What's covered

**Tools and concepts**

- Nextflow, nf-core, GCP Batch, viralrecon — what each one is
- Why notebook 03 pins specific versions of Nextflow, JDK, and viralrecon
- The `-resume` flag and how Nextflow's cache works

**File formats**

- FASTQ: raw sequencing reads plus quality scores
- FASTA and consensus FASTA: nucleotide sequences without quality
- VCF: variant calls relative to a reference
- BAM and SAM: aligned reads
- GFF and GTF: genome feature annotations
- BED: simple genomic intervals

**Pipeline conventions**

- Samplesheet: the CSV nf-core pipelines take as input
- viralrecon output directory layout
- nf-core profiles and config layers

A condensed reference table sits at the bottom.

## Tools and concepts

**Nextflow** is a workflow language. You describe a pipeline as a graph
of processing steps; Nextflow handles scheduling, where things run, and
how inputs and outputs flow between steps. It runs on the JVM, which is
why notebook 03 installs JDK 17 alongside it.

**nf-core** is a community-maintained catalog of canonical Nextflow
pipelines. Each one is tested, documented, and configurable through a
standard set of profiles and parameters. `nf-core/viralrecon` is one;
`nf-core/sarek`, `nf-core/rnaseq`, and dozens of others follow the same
conventions.

**GCP Batch** is Google Cloud's managed batch-compute service. Nextflow
submits each pipeline step as a Batch job, which spins up its own VM,
runs the step's container, writes output to a shared bucket, and shuts
down. Scales up and down with the pipeline.

**viralrecon** is one specific nf-core pipeline, built for paired-end
short-read viral whole-genome sequencing. The stages it runs:

- QC with FastQC, one report per sample
- Trimming and quality filtering with fastp
- Alignment to a viral reference (BWA or Bowtie2)
- Variant calling with iVar or BCFtools
- Per-sample consensus FASTA generation
- A MultiQC summary aggregating everything into one HTML report
- For SARS-CoV-2 data, lineage assignment via Pangolin and NextClade

## Why pinned versions matter

Notebook 03 pins three things: Nextflow at `23.10`, JDK at `17`, and
viralrecon at `2.6.0`. Each pin exists because something newer broke
something older.

- **Nextflow 24+** ships strict DSL2 mode by default and rejects older
  nf-core configs (including viralrecon 2.6.0). The 23.10 LTS branch
  is the last one fully compatible with viralrecon 2.6.0, and it's
  still receiving patches.
- **JDK 17** is what Nextflow 23.10 ships against. JDK 21+ surfaces
  reflective-access warnings that look alarming in the logs even
  though the pipeline runs fine.
- **viralrecon 2.6.0** is the most recent release where the bundled
  `conf/test.config` references files that are actually shipped in
  the release archive. The pipeline's main branch sometimes references
  configs that haven't been published yet.

Default to the pinned set for reproducibility. If you need to upgrade,
expect to chase config syntax changes across all three at once.

## The `-resume` flag

Every Nextflow command in notebook 03 uses `-resume`. If a previous run
completed some steps before failing or being cancelled, Nextflow reuses
those cached results instead of re-running them. The cache lives in the
`WORK_BUCKET` you configured.

`-resume` is almost always what you want. The one exception: an input
file or parameter changed but Nextflow's cache key didn't notice (rare,
but happens). In that case, drop `-resume` to force every step to
re-execute. Deleting the work directory has the same effect.

## FASTQ

The raw output of a short-read sequencer. Every read is four lines:

```
@SRR13000260.1 1 length=150
ACGTAGCTAGCTAGCATCGTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCATGCAT...
+
IIIIIIIIIHHGGFFEEDDDCCBBBAAA9988776655443322110//..--,,,++**)
```

1. **Identifier**: starts with `@`. Often includes run, lane, flowcell,
   and read number from the sequencer. Format varies by platform.
2. **Sequence**: the nucleotide bases the sequencer called. Length is
   fixed within an Illumina run, variable on long-read platforms.
3. **Separator**: a literal `+`. Sometimes has the identifier repeated
   after it, often doesn't.
4. **Quality string**: one ASCII character per base, same length as the
   sequence. Each character encodes a Phred quality score.

Files are usually gzip-compressed (`.fastq.gz`). Most tools read the
compressed form directly; you don't need to decompress first.

### Phred quality scores

A Phred score Q encodes the probability `P` that a base call is wrong:

```
Q = -10 × log10(P)
```

So:

| Q  | P (error rate) | Accuracy |
|----|----------------|----------|
| 10 | 1 in 10        | 90%      |
| 20 | 1 in 100       | 99%      |
| 30 | 1 in 1000      | 99.9%    |
| 40 | 1 in 10000     | 99.99%   |

Illumina aims for most bases at Q30 or above. Quality tends to drop
near the 3' end of each read, which is why trimming exists.

The ASCII encoding (Phred+33) maps `!` (ASCII 33) to Q0, `"` to Q1,
through `I` at Q40 and beyond. In the example above, the `I` runs are
Q40 (excellent) and the tail bases at `)` and `*` are around Q8-Q9
(poor — fastp would likely trim them).

### Paired-end naming

Paired-end runs split each sample across two FASTQ files:

- `<sample>_1.fastq.gz` (or `_R1`): forward reads
- `<sample>_2.fastq.gz` (or `_R2`): reverse reads

The two files have **identical read counts**. Read N in `_1` matches
read N in `_2`. They came from opposite ends of the same DNA fragment,
so a mismatch points to corruption or truncation.

Both `_1`/`_2` and `_R1`/`_R2` conventions exist. APGAP datasets use
`_1`/`_2`. nf-core pipelines accept either; the samplesheet decouples
the filename from the column meaning.

### Inspecting a FASTQ file

Three quick ways, in increasing order of formality:

**Raw, with gzip:**

```python
import gzip
with gzip.open("sample_1.fastq.gz", "rt") as f:
    for i, line in enumerate(f):
        if i >= 8: break  # 2 records
        print(line.rstrip())
```

**With Biopython:**

```python
from Bio import SeqIO
import gzip
with gzip.open("sample_1.fastq.gz", "rt") as f:
    for record in SeqIO.parse(f, "fastq"):
        print(record.id, len(record.seq))
        break
```

**From the shell:**

```
zcat sample_1.fastq.gz | head -8
```

Notebook 02 walks through inspection patterns end to end, including
streaming from GCS without downloading the whole file first.

## FASTA and consensus FASTA

FASTA is FASTQ minus the quality scores. Each record is two parts:

```
>sample123 Influenza A, H1N1, segment 4
ACGTAGCTAGCTAGCATCGTAGCTAGCTAGCATGCATGCATGCATGCAT
GCATGCATGCATGCATGCATGCATGCATGCATGCAT...
```

- **Header**: starts with `>`. Free-form description. The first
  whitespace-separated token is conventionally the sequence ID.
- **Sequence**: nucleotide or amino acid characters, often wrapped
  across lines at 60 or 80 columns.

Multi-record FASTA files repeat the header-sequence pattern. They're
plain text by default; gzip variants are common too (`.fasta.gz`,
`.fa.gz`).

### Consensus FASTA

A consensus FASTA is what comes out of variant calling. After aligning
reads to a reference and identifying every position where the sample
differs, the consensus is the reference with those differences applied.
One sequence per sample.

viralrecon writes consensus FASTAs as `<sample>.consensus.fa` under the
variant-calling output subdirectory. These are what you would submit to
GISAID or NCBI as the assembled genome for a sample.

Low-coverage positions are masked with `N` so downstream tools
(phylogenetic inference, lineage assignment) know not to trust them.
A consensus full of `N`s usually means the sample had patchy coverage.

## VCF

VCF (Variant Call Format) is what variant callers emit. Each row is one
position where one or more samples differ from a reference. Tab-
separated, with a header block of `##`-prefixed metadata lines and one
`#`-prefixed column header line:

```
##fileformat=VCFv4.2
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
#CHROM  POS    ID  REF  ALT  QUAL  FILTER  INFO        FORMAT  Sample1
chr1    14653  .   C    T    100   PASS    DP=42       GT      0/1
chr1    14907  .   A    G    85    PASS    DP=37       GT      1/1
```

Key columns:

- **CHROM, POS**: position on the reference
- **REF, ALT**: reference base and the observed alternative(s)
- **QUAL**: a Phred-scaled confidence score for the call
- **FILTER**: `PASS` or the name of a filter the variant failed
- **INFO**: site-level annotations, defined in the `##INFO` header
- **FORMAT + sample columns**: per-sample fields (genotype, depth, etc.)

Almost always gzipped and indexed: `.vcf.gz` next to `.vcf.gz.tbi`. The
index lets tools jump to a region without scanning the whole file.

### Reading VCF in Python

```python
import gzip
with gzip.open("variants.vcf.gz", "rt") as f:
    for line in f:
        if line.startswith("#"): continue
        fields = line.rstrip().split("\t")
        chrom, pos, _, ref, alt = fields[:5]
        print(f"{chrom}:{pos} {ref}>{alt}")
```

For anything past simple scanning, use `pysam.VariantFile` or `cyvcf2`.
Both handle the header schema and per-sample fields properly, and both
can pull records from a specific region using the `.tbi` index.

## BAM and SAM

Aligned reads. SAM is the human-readable text form, BAM is the binary
equivalent (smaller, faster, what tools actually consume). Every aligned
read is one row with columns for the read name, where it mapped,
mapping quality, a CIGAR string describing the alignment, the sequence,
and the qualities.

You almost never read these by hand. Tools that look at them:

- `samtools view file.bam | head`: dump the first few alignments
- `samtools flagstat file.bam`: summary counts (mapped, paired, etc.)
- `pysam.AlignmentFile`: parse in Python

viralrecon writes per-sample BAMs in the alignment subdirectory, indexed
(`.bam.bai`) so genome browsers can load specific regions without
reading the whole file.

## GFF and GTF

Genome feature annotations. Each row describes one feature (gene, exon,
CDS, regulatory region) with its position on a reference. Used by
variant annotators (snpEff, VEP) to translate variants into biological
consequences ("this SNP changes amino acid 614 of the spike protein").

Tab-separated with a `#`-prefixed header. The two formats are similar
enough that tools generally accept both. You'll see GFF3 referenced by
nf-core pipelines that need annotation; viralrecon uses one when it
runs snpEff over its variant calls.

## BED

A simple format for "here are some intervals on a reference":

```
chrom    start  end    name    score  strand
chr1     65    75     region1 100    +
chr1    100   200     region2 200    -
```

Three columns minimum (`chrom`, `start`, `end`); more are optional. Used
for primer schemes (which positions to clip), masking, capture targets,
and lots of other "list of intervals" purposes. viralrecon's amplicon
protocol reads a BED of primer positions.

## Samplesheet

nf-core pipelines take a CSV called a samplesheet as their primary
input. The exact columns vary by pipeline. For `nf-core/viralrecon`
with Illumina paired-end data:

```
sample,fastq_1,fastq_2
SRR13000260,gs://my-bucket/SRR13000260_1.fastq.gz,gs://my-bucket/SRR13000260_2.fastq.gz
SRR13000261,gs://my-bucket/SRR13000261_1.fastq.gz,gs://my-bucket/SRR13000261_2.fastq.gz
```

Rules:

- **One row per sample, not per file**. Paired-end FASTQs go in two
  columns on the same row.
- **`sample` is your label, not the FASTQ filename**. It becomes the
  prefix on every output for that sample. Stick to alphanumerics and
  underscores; avoid spaces and special characters.
- **Paths can be local or `gs://` URIs**. Mixing both in one sheet
  works too.
- **Extra columns are ignored** by viralrecon. Some pipelines do read
  them (`nf-core/sarek` reads `patient`, `status`, `lane`; check the
  pipeline's docs).

Notebook 03 has a cell that builds one automatically from whatever's
in your dataset bucket.

## viralrecon output directory layout

After a successful viralrecon run, the `--outdir` you passed contains
something like:

```
viralrecon-results/
  ├── fastqc/             # per-sample raw and trimmed FastQC HTML + zip
  ├── fastp/              # trimming reports
  ├── bowtie2/            # alignment BAMs
  │   └── log/            # alignment summary text
  ├── variants/
  │   ├── ivar/           # variant calls + consensus FASTA per sample
  │   │   ├── *.tsv       # raw iVar output
  │   │   ├── *.vcf.gz    # converted to VCF
  │   │   └── consensus/  # per-sample .consensus.fa
  │   └── snpeff/         # variants annotated with biological consequences
  ├── multiqc/
  │   └── multiqc_report.html  # start here
  └── pipeline_info/      # execution report, DAG, trace
```

Exact subdirectory names depend on the choices viralrecon made (Bowtie2
vs BWA, iVar vs BCFtools). The MultiQC report is the natural entry
point: it aggregates every other step's metrics into one self-contained
HTML page.

## nf-core profiles and config layers

A Nextflow run combines four sources of configuration, in this order
(later wins):

1. **Pipeline defaults**: shipped with the pipeline in `conf/`.
2. **Profile**: a named bundle of overrides, e.g. `test` (synthetic
   data) or `gcb` (the GCP Batch profile notebook 03 writes).
3. **`-c <file>`**: an extra config you pass on the command line. This
   is where notebook 03's `nextflow.config` plugs in.
4. **`--param value`** flags: highest precedence.

Profiles can stack with commas: `-profile test,gcb` applies both. That
combination is how notebook 03's first launch runs the bundled test
data on GCP Batch.

## Quick reference

| Format    | Suffix(es)            | What it is                              | Read with                      |
|-----------|-----------------------|------------------------------------------|--------------------------------|
| FASTQ     | `.fastq`, `.fastq.gz` | Raw reads plus per-base qualities        | `Bio.SeqIO`, `gzip`            |
| FASTA     | `.fa`, `.fasta`       | Sequences only, no qualities             | `Bio.SeqIO`                    |
| VCF       | `.vcf`, `.vcf.gz`     | Variants vs a reference                   | `pysam.VariantFile`, `cyvcf2`  |
| BAM       | `.bam` (+ `.bai`)     | Aligned reads, binary                    | `pysam.AlignmentFile`          |
| SAM       | `.sam`                | Same as BAM but text                     | `samtools view`                |
| GFF/GTF   | `.gff`, `.gtf`        | Feature annotations on a reference        | `gffutils`, plain TSV          |
| BED       | `.bed`                | Genomic intervals                         | plain TSV                      |

Compressed variants (`.gz`) tend to ship with a companion index file
(`.tbi` for VCF, `.bai` for BAM) so tools can seek without
decompressing the whole thing.

## Going deeper

Each format has a spec document; in practice you rarely need to read
them directly. The Wikipedia pages are usually accurate enough:

- [FASTQ format](https://en.wikipedia.org/wiki/FASTQ_format)
- [VCF specification](https://samtools.github.io/hts-specs/VCFv4.5.pdf) (htsspec PDF)
- [SAM/BAM specification](https://samtools.github.io/hts-specs/SAMv1.pdf) (htsspec PDF)
- [GFF3 specification](https://gmod.org/wiki/GFF3)
- [Biopython SeqIO reference](https://biopython.org/wiki/SeqIO)
- [nf-core pipeline standards](https://nf-co.re/docs/contributing/pipelines)

Back to the [getting-started overview](01-getting-started.ipynb).